# Middleware Router
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Strings, Tries · **Difficulty/Frequency:** Very Common (8/10)


## Concepts

**What this problem is really testing:**
- A trie, but keyed by path *segment* instead of by character
- Depth-first search with a clear rule for "which branch to try first"
- The general idea of "the most specific match wins"

**Why each one shows up here:**
- A URL path is naturally hierarchical — `/bar/a/baz` is really the segments `["bar", "a", "baz"]`. A trie exists exactly for this: mirroring a hierarchical key so a shared prefix (a shared leading path) is only stored once.
- Wildcards add a second kind of edge out of each node, besides the normal "literal segment" edges. That turns lookup into a small DFS that has to try both kinds of edge, in a specific order.

**The one idea to hold onto:** this is the same trie idea used for autocomplete — just swap "characters" for "path segments" — plus one extra rule: at every node, try the literal child first, and only fall back to the wildcard child if the literal path leads nowhere.

---

### Quick primers — the building blocks used below

**What is a Trie (prefix tree)?**
- A trie is a tree where each edge is labeled with one piece of a key — a character, or here, a path segment.
- Following a path from the root down spells out a stored key, one edge-label at a time.
- Shared prefixes are stored once (as shared ancestor nodes) — that's what makes it efficient for "does any key start with this?" and exact lookups.
- **Cost:** insert and exact lookup are both O(L), where L is the length of the key (in segments here) — this doesn't depend on how many *other* keys are stored.
- **In Python:** usually built from nested `dict`s — a node is just a `dict` mapping "next unit" → "next node". That's the representation used below.

**Depth-first search (DFS) with a precedence rule.**
- DFS explores one path as far as it can go before backtracking to try something else.
- Here, each node has up to two options: the literal child for the current segment, or the wildcard child.
- "DFS with precedence" means: fully explore the literal option first, and only try the wildcard option if the literal path comes back empty-handed.
- This one rule is exactly how the router decides that `/foo/baz` beats `/foo/*`.


## Problem Statement

```python
Router.addRoute("/bar", "result")
Router.callRoute("/bar")   # -> "result"
```

**Follow-up -- wildcard `*` matches any single segment:**

```python
router.addRoute("/foo", "foo")
router.addRoute("/bar/*/baz", "bar")
router.callRoute("/bar/a/baz")   # -> "bar"
```

**Discussion question:** given

```python
router.addRoute("/foo/baz", "foo")
router.addRoute("/foo/*", "bar")
```

which should `callRoute("/foo/baz")` return -- the exact match `"foo"`, or the wildcard match `"bar"`? (Answered explicitly below: **exact matches win**.)


### Approach 1 -- Naive (flat exact-match dict)

**Idea:** the simplest thing that satisfies the base usage example -- a `dict` from the full path string to its result.

**Time complexity:** O(1) lookup ... for exact routes only.

**Space complexity:** O(N) for N registered routes.

**Why it's not enough:** it has no notion of "segments", so it cannot represent -- let alone match -- a wildcard route like `/bar/*/baz`. This is a correctness gap, not a performance one: the moment the wildcard follow-up appears, this approach simply cannot express the requirement.


In [ ]:
from typing import Dict, List, Optional


class RouterNaive:
    def __init__(self) -> None:
        self.routes: Dict[str, str] = {}

    def addRoute(self, path: str, result: str) -> None:
        self.routes[path] = result             # no concept of segments or wildcards

    def callRoute(self, path: str) -> Optional[str]:
        return self.routes.get(path)           # exact string match only


### Approach 2 -- Optimal (segment trie, literal-first DFS)

**Idea:** split every path into segments on `/`. Build a trie where each node has a `children` dict (literal segment -> next node) and an optional `'*'` key (wildcard -> next node); a node with a stored `result` marks "a route terminates here". `callRoute` walks segment by segment: at each node, recurse into the **literal** child first; only if that subtree returns nothing (`None`) do you fall back to the **wildcard** child. This gives exact matches priority over wildcards at *every* level, which is exactly what resolves the discussion question: `/foo/baz` (all-literal path) always finds the literal route before the DFS ever considers `/foo/*`.

**Time complexity:** O(L) per `callRoute`, where L is the number of segments in the queried path -- the literal and wildcard branches at a given node are tried in sequence, but each level of the trie is visited at most a small constant number of times (one literal attempt, one wildcard attempt), so total work is proportional to path depth, not to the number of registered routes.

**Space complexity:** O(sum of segment counts across all registered routes) -- shared prefixes (e.g. many routes under `/api/...`) share trie nodes, same as a character-level trie sharing prefixes.


In [ ]:
class Router:
    def __init__(self) -> None:
        self.root: dict = {}

    def addRoute(self, path: str, result: str) -> None:
        segments = path.strip("/").split("/")
        node = self.root
        for seg in segments:
            if seg == "*":
                node = node.setdefault("*", {})
            else:
                node = node.setdefault("children", {}).setdefault(seg, {})
        node["result"] = result

    def callRoute(self, path: str) -> Optional[str]:
        segments = path.strip("/").split("/")
        return self._match(self.root, segments, 0)

    def _match(self, node: dict, segments: List[str], index: int) -> Optional[str]:
        if index == len(segments):
            return node.get("result")

        seg = segments[index]

        if "children" in node and seg in node["children"]:      # try the LITERAL branch first
            result = self._match(node["children"][seg], segments, index + 1)
            if result is not None:
                return result

        if "*" in node:                                          # fall back to wildcard only if literal failed
            result = self._match(node["*"], segments, index + 1)
            if result is not None:
                return result

        return None


### Discussion question, resolved

Registering `/foo/baz -> "foo"` then `/foo/*  -> "bar"` and calling `callRoute("/foo/baz")`: at the node for segment `"foo"`, the walk tries the **literal** child for `"baz"` first, finds it, and it has `result = "foo"` stored at the end -- so the literal branch returns `"foo"` immediately, and the wildcard branch (`/foo/*`) is **never even tried**. The exact match wins, by construction of the literal-first ordering -- verified below.


## Verification

Build the routers from every example in the problem, confirm behavior, and check the edge cases each Talking Point calls out.

In [ ]:
# --- Base usage ---
router = Router()
router.addRoute("/bar", "result")
assert router.callRoute("/bar") == "result"
assert router.callRoute("/nope") is None

naive = RouterNaive()
naive.addRoute("/bar", "result")
assert naive.callRoute("/bar") == "result"

# --- Wildcard follow-up ---
router2 = Router()
router2.addRoute("/foo", "foo")
router2.addRoute("/bar/*/baz", "bar")
assert router2.callRoute("/foo") == "foo"
assert router2.callRoute("/bar/a/baz") == "bar"
assert router2.callRoute("/bar/anything/baz") == "bar"   # wildcard matches ANY single segment
assert router2.callRoute("/bar/a/b/baz") is None          # wildcard matches exactly ONE segment, not two
assert router2.callRoute("/bar/a/wrong") is None

# --- Discussion question: exact match must beat wildcard ---
router3 = Router()
router3.addRoute("/foo/baz", "foo")
router3.addRoute("/foo/*", "bar")
assert router3.callRoute("/foo/baz") == "foo"    # exact wins
assert router3.callRoute("/foo/qux") == "bar"    # no exact match for "qux" -> wildcard catches it

# --- Root path edge case ---
router4 = Router()
router4.addRoute("/", "home")
assert router4.callRoute("/") == "home"          # strip("/") on both "/" -> [""] consistently

# --- Order of addRoute calls shouldn't matter for this precedence rule ---
router5 = Router()
router5.addRoute("/foo/*", "bar")     # wildcard registered FIRST this time
router5.addRoute("/foo/baz", "foo")
assert router5.callRoute("/foo/baz") == "foo"    # still exact-wins, regardless of registration order

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Named parameters, like `/users/:id/posts`.** Store the parameter name alongside a wildcard-like node (e.g. a `params` dict keyed by placeholder name -> the node), and have `_match` collect `{"id": segment_value}` as it descends, returning it alongside the result. The same literal-first precedence rule applies: `/users/:id` and `/users/me` can coexist, with `/users/me` winning for that literal path. A minimal version is implemented below.
- **A `**` that matches zero or more segments.** This breaks the "one edge per segment" model -- a `**` node would need to try consuming 0, 1, 2, ... remaining segments and recurse for each count, which reintroduces the exponential-blowup risk the Talking Points warn about if not memoized/pruned carefully. Worth naming as a materially harder extension rather than a small tweak.
- **Thread-safety.** If routes are only ever added during startup and the trie is read-only afterward, no locking is needed for `callRoute` -- concurrent reads of an immutable structure are always safe. Locking only becomes necessary if `addRoute` can be called while `callRoute` is concurrently in flight (dynamic route registration).
- **Memory overhead vs. a flat dictionary.** A trie adds one node per *unique path prefix*, not per route -- so it's cheaper than a flat dict when routes share prefixes heavily (e.g. a deep `/api/v1/...` hierarchy), and only loses when routes are mostly unrelated single-segment paths, where the per-node dict overhead isn't offset by any sharing.
- **Route registration order and conflicts.** The verification above confirms this implementation is **order-independent** for literal-vs-wildcard precedence -- because precedence is decided at *lookup* time (always try literal first), not at *registration* time. The one real conflict case is registering the *exact same* path twice: the second `addRoute` call silently overwrites the first's `result`, which is worth calling out explicitly as a policy decision (last-write-wins) rather than leaving it implicit.


In [ ]:
class RouterWithParams(Router):
    """Bonus: adds named-parameter segments like :id, captured and returned alongside the result."""

    def addRoute(self, path: str, result: str) -> None:
        segments = path.strip("/").split("/")
        node = self.root
        for seg in segments:
            if seg == "*":
                node = node.setdefault("*", {})
            elif seg.startswith(":"):
                node = node.setdefault("param", {"name": seg[1:], "node": {}})["node"]
            else:
                node = node.setdefault("children", {}).setdefault(seg, {})
        node["result"] = result

    def callRouteWithParams(self, path: str):
        segments = path.strip("/").split("/")
        return self._match_params(self.root, segments, 0, {})

    def _match_params(self, node: dict, segments: List[str], index: int, params: dict):
        if index == len(segments):
            return (node.get("result"), params) if "result" in node else None

        seg = segments[index]

        if "children" in node and seg in node["children"]:       # literal still wins first
            out = self._match_params(node["children"][seg], segments, index + 1, params)
            if out is not None:
                return out

        if "param" in node:                                       # then named parameter
            new_params = {**params, node["param"]["name"]: seg}
            out = self._match_params(node["param"]["node"], segments, index + 1, new_params)
            if out is not None:
                return out

        if "*" in node:                                           # wildcard last
            out = self._match_params(node["*"], segments, index + 1, params)
            if out is not None:
                return out

        return None


rp = RouterWithParams()
rp.addRoute("/users/:id/posts", "user-posts")
result, params = rp.callRouteWithParams("/users/42/posts")
assert result == "user-posts" and params == {"id": "42"}
print("Named-parameter routing works:", result, params)


## Empirical complexity check

`callRoute` is O(L) in the number of **path segments**, independent of how many routes are registered. We register a fixed small set of routes at varying **depths** and confirm lookup time scales with path length L, not with N (registered-route count, held fixed).

| Growth when L doubles | Implies |
|---|---|
| ~2x | linear in path depth |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

sys.setrecursionlimit(10000)   # _match recurses once per path segment -- give it headroom


def make_worst_case(depth):
    # One deep chain of literal segments, ending in a result -- callRoute must walk the
    # full depth with no early exit (every prefix segment matches, only the LAST decides).
    r = Router()
    path = "/" + "/".join(f"seg{i}" for i in range(depth))
    r.addRoute(path, "leaf")
    return (r, path)


def call_deep_route(router, path):
    return router.callRoute(path)


solutions = {"callRoute (trie, O(L))": call_deep_route}
sizes = [200, 400, 800, 1600]     # path depth L -- unrealistically deep for a real router,
                                   # but large enough to see the O(L) trend over timing noise
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Trie the key by whatever your natural "unit" is, not just characters.** Here the unit is a path segment; the same idea keys a trie by words (autocomplete phrases) or by any other hierarchical token.
- **Encode precedence in traversal order, not in the data.** "Exact beats wildcard" doesn't need a priority field stored anywhere -- it falls out of simply trying the literal branch before the wildcard branch in `_match`, at every level, making the rule impossible to violate accidentally.
- **A DFS with an ordered fallback is a clean way to express "try the specific thing, then the general thing".** The same shape shows up in CSS selector specificity, route matching in web frameworks, and overload resolution in some type systems.
- **Order-independence is a correctness property worth testing for explicitly.** Because precedence is resolved at lookup time here, registration order doesn't matter -- but that's a property of *this* design, not something to assume for free; the verification cell tests it directly rather than trusting it by inspection.
- **Related problems:** Implement Trie (prefix tree), Word Search II (trie + DFS over a grid), Add and Search Word (trie with a wildcard `.` -- nearly identical precedence-DFS shape to this problem).
- **Common pitfalls:** trying the wildcard branch first (silently inverts the precedence rule); forgetting that a subtree can partially match then dead-end, requiring the caller to backtrack and try the *other* branch at a shallower level (handled here by returning `None` and letting the caller's `if result is not None` decide); not handling the root path (`"/"` splitting to `[""]`) consistently with how routes are registered.
